# 16 · Interface de análise de sobrevivência (`SurvivalUI`) para a PD lifetime

O tutorial **13** mostrou os motores da estrutura a termo (`yggdrasil.credit_risk.ecl`): a curva de
safra, o Kaplan-Meier, o *hazard* em tempo discreto e a fachada `LifetimePD`. Este tutorial mostra a
**bancada de trabalho** em cima deles: o subpacote `yggdrasil.credit_risk.survival`, com as ferramentas
que a análise de sobrevivência usa para **construir, escolher e defender** uma curva, e a interface
`SurvivalUI`, que conduz o estudo aba a aba no mesmo desenho das demais UIs do `credit_risk`.

O que o subpacote acrescenta ao `ecl`:

| bloco | o que responde |
|---|---|
| tabela de vida, Nelson-Aalen, **log-rank** | quantos em risco, quantos censurados, e as curvas dos segmentos diferem? |
| famílias **paramétricas** (Weibull, log-normal, log-logística, Gompertz, exponencial) | qual a forma da maturação, e o que a curva faz **além da última idade observada**? |
| **emenda** empírica + cauda paramétrica | a curva até o horizonte do contrato sem repetir o último hazard |
| **validação** (backtest com z de Greenwood, AUC/KS por horizonte, **C-index**, decil com Hosmer-Lemeshow, riscos proporcionais) | a curva acerta o nível, ordena o risco e a hipótese do modelo sobrevive aos dados? |
| `SurvivalConfig` / `run_survival_study` | o estudo inteiro em JSON, reproduzível fora do notebook |

O fluxo: **painel de contratos → curva (KM · hazard · paramétrica) → cauda → calibração e ciclo →
validação → `LifetimePD` para o ECL**.

## 0. Setup

A interface só precisa de `ipywidgets` na sessão (extra `[ui]`) e de um painel longo em memória.
`%matplotlib inline` porque a UI devolve as figuras já embutidas como imagem.

In [ ]:
# --- Bootstrap: torna o pacote `yggdrasil` importável a partir do repositório,
# sem `pip install`. Procura a raiz do repo (a pasta que contém `yggdrasil/`) por
# vários âncoras — o caminho do próprio notebook (VS Code expõe `__vsc_ipynb_file__`),
# o diretório atual, os diretórios do sys.path e, no Databricks, o caminho via
# dbutils — subindo até achá-la, e a insere no sys.path. Cobre Jupyter/VS Code
# local e Databricks; se o pacote já estiver importável, é inócuo.
import sys
from pathlib import Path

def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None

_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))

In [ ]:
%matplotlib inline
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")   # MLflow local (seção 9)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

from yggdrasil.credit_risk.survival import (
    SurvivalUI, SurvivalConfig, run_survival_study, make_reference_panel,
    life_table, logrank_test, fit_parametric, splice_curves, junction_age,
    backtest_curve, discrimination_by_horizon, model_concordance, calibration_by_decile, ph_test,
)
from yggdrasil.credit_risk.ecl import ContractPanel, LifetimePD, kaplan_meier

## 1. Os dados: o painel longo de contratos

O contrato da interface é o mesmo do `ContractPanel`: **uma linha por contrato × safra de observação**,
com a flag de **entrada em default** no período. A idade (*months on book*) pode vir pronta, ser
derivada da safra de originação ou da posição na trajetória.

Sem base em mãos, o subpacote traz um **painel de referência** com processo gerador conhecido: dois
produtos, três covariáveis (`feat_score`, `feat_ltv`, `feat_atraso_prev`), **maturação** de Weibull
(`k = 1,35`, hazard crescente com a idade) e **censura informativa** (quem tem menos risco quita mais
cedo, como na carteira real). É ele que permite checar se os motores recuperam a verdade.

In [ ]:
ref = make_reference_panel(n_contracts=1500, months=36, seed=7)
df = ref.df
print(ref)
print("hazard verdadeiro (cartão, score médio) nas idades 0, 12, 24, 35:",
      np.round(ref.true_hazard([0, 12, 24, 35], produto="cartao"), 4))
df.head()

### 1.1. Plugando os **seus** dados

No caso real o painel vem de uma consulta ao histórico de contratos. Passe o `DataFrame` e o mapeamento
de colunas no construtor (ou ajuste os dropdowns da aba **Painel** e clique em *Montar painel*):

```python
ui = SurvivalUI(
    meu_painel,
    id_col="id_contrato", date_col="dt_ref", default_col="flag_default",
    origin_col="dt_originacao",          # a idade é derivada daqui (ou passe age_col=...)
    term_col="prazo_remanescente", segment_col="produto", exposure_col="saldo",
    features=["score", "ltv", "atraso_max_6m"],   # numéricas, sem NaN
    horizon=60,
)
```

Regras que valem sob a 4.966: a flag é o **evento** (o painel reduz o *estado* ao evento descartando
observações posteriores ao primeiro default), pares `(contrato, safra)` duplicados são erro, e
categóricas entram já codificadas (a política de categorização é a do `ModelSegmenter`).

## 2. Abrindo a interface

Instancie e deixe o objeto como **última linha** da célula. Daqui em diante o trabalho é na tela; o
notebook segue comentando o que olhar em cada aba e **reproduzindo os cliques em código**, para que o
tutorial possa ser lido inteiro sem uma sessão interativa.

In [ ]:
ui = SurvivalUI(
    df,
    origin_col="safra_origem", segment_col="produto", term_col="prazo", exposure_col="exposicao",
    features=list(ref.features), by="produto", horizon=60, name="pd_lifetime_referencia",
)
ui   # exibe o workbench de 7 abas

## 3. Aba 1. **Painel**: o retrato da carteira e a partição DES/OOT

**O que olhar.**

1. **Os mosaicos e o gráfico de base em risco.** A base em risco é recontada **idade a idade**: os
   contratos jovens entram nas idades baixas e simplesmente não aparecem nas altas. A barra cinza é a
   censura por idade (fim de janela, quitação, prazo); a linha vermelha é o hazard observado. Quando a
   base cai abaixo de algumas dezenas de contratos, o hazard fica serrilhado: é ali que a curva
   empírica deixa de ser confiável e a cauda paramétrica (aba 4) assume.
2. **O mapa safra × idade.** Maturação é andar para a direita; qualidade da safra é mudar de linha.
3. **A partição.** A curva é ajustada no DES e validada no OOT. Por **safra de originação** é a
   validação mais exigente (contratos que o ajuste nunca viu); por **data de observação** testa a curva
   a partir das idades em que a carteira viva está. A data de corte sugerida deixa o último terço para o
   OOT.

In [ ]:
# = clicar "Desenhar mapa safra × idade"
ui.dd_cohort.value = "Q"
ui.btn_heat.click()

# = escolher "por safra de originação" e clicar "Particionar"
ui.dd_split.value = "origin"
print("corte sugerido:", ui.tx_split_date.value)
ui.btn_split.click()
print(ui.des_, "|", ui.oot_)

## 4. Aba 2. **Kaplan-Meier**: a curva que a carteira mostrou

**O que olhar.**

- **Kaplan-Meier e a curva de safra coincidem no ponto estimado** (em tempo discreto ambos são o
  produto-limite $\prod (1 - d_t/n_t)$). O KM acrescenta a incerteza (erro padrão de **Greenwood**, IC
  log-log); a safra acrescenta a base mínima por idade. **Nelson-Aalen** estima o hazard acumulado
  $\sum d_t/n_t$ e fica sempre um pouco acima do KM: onde os dois se afastam, a base rareia.
- As **quatro representações** (acumulada, sobrevivência, hazard, marginal) são a mesma curva.
- A tabela resume por grupo: PD de 12 meses, PD até a última idade observada, a **mediana** (período em
  que $S(t)$ cruza 50%) e o **RMST** (tempo médio de sobrevivência restrito ao horizonte).
- **Log-rank**: H0 = as curvas dos grupos são iguais. Rejeitar é o argumento para curvas **por
  segmento**; não rejeitar sugere uma curva única (ou fundir os pares que não se distinguem, no teste
  par a par com Bonferroni).

In [ ]:
# = escolher agrupar por "produto", representação "PD acumulada" e clicar "Estimar curvas"
ui.dd_km_by.value = "produto"
ui.dd_km_kind.value = "cumulative"
ui.cb_km_ci.value = True
ui.btn_km.click()

lr = ui.logrank_
print(f"log-rank: chi2 = {lr['estatistica']:.2f}, gl = {lr['gl']}, p = {lr['p_valor']:.2e}")
display(lr["grupos"])
ui.life_table_.query("grupo == 'cartao'").head(8)

In [ ]:
# A mesma conta sem a interface (o que a aba faz por trás):
painel_des = ui.des_
tab = life_table(painel_des, by="produto", alpha=0.05)
print(tab.columns.tolist())
logrank_test(painel_des, "rating")["p_valor"]

### 4.1. Adotar a curva

Qualquer aba **adota** a curva que produziu como a curva do estudo; a partir daí Calibração, Validação
e Exportar operam sobre ela. Adotar do KM dá uma curva com **cauda plana** (repete o último hazard até o
horizonte): é a hipótese mínima, e a aba Paramétrico existe para trocá-la.

In [ ]:
ui.sl_km_horizon.value = 60
ui.btn_km_adopt.click()
print(ui.model_, "| origem:", ui.model_source_)
ui.model_.summary()

## 5. Aba 3. **Hazard**: a curva por contrato

A regressão de *hazard* em tempo discreto expande o painel em **pessoa-período** e ajusta
$P(\text{quebra em } t \mid \text{vivo em } t) = g^{-1}(f(\text{idade}) + x\beta)$: a idade entra como
**baseline** (a maturação), as covariáveis deslocam a linha de base. A curva passa a ser **por
contrato**, que é o que o ECL por contrato pede.

**O que olhar.**

- **Coeficientes e odds ratio.** $e^{\beta}$ é quanto a chance condicional de quebrar em cada período
  multiplica por unidade da feature. No painel de referência, `feat_score` deve sair positivo e perto do
  $\beta$ verdadeiro (0,55).
- **Contrato médio × KM.** Se a curva do modelo se afasta do KM em idades com base grande, o problema é
  do *baseline* (`spline` suaviza a cauda; `dummies` é totalmente flexível).
- **Perfis P10 / P50 / P90.** A abertura entre eles é o quanto as covariáveis discriminam.
- **Riscos proporcionais.** H0: o efeito de cada covariável é o mesmo em todas as idades. O teste
  compara, por razão de verossimilhança, o modelo contra o mesmo modelo com interações
  `feature × ln(1 + idade)`. O painel de referência é proporcional por construção: o teste não deve
  rejeitar.

In [ ]:
# = marcar as features, baseline "spline" e clicar "Ajustar hazard"; depois "Testar riscos proporcionais"
ui.sel_hz_features.value = ("feat_score", "feat_ltv", "feat_atraso_prev")
ui.dd_hz_baseline.value = "spline"
ui.dd_hz_by.value = None                       # um modelo único para a carteira (o produto NÃO entra: ver a armadilha abaixo)
ui.sl_hz_horizon.value = 60
ui.btn_hz.click()
ui.btn_hz_ph.click()

mh = ui.hazard_lt_.hazard_models_["__global__"]
coef = mh.coef_frame()
display(coef[coef["termo"].isin(["(intercepto)", "feat_score", "feat_ltv", "feat_atraso_prev"])])
print("beta verdadeiro:", ref.truth["beta"])
print(f"riscos proporcionais: p = {ui.ph_['p_valor']:.3f} -> {'não rejeita' if ui.ph_['proporcional'] else 'rejeita'} H0")

> **Armadilha metodológica.** O produto não entrou como covariável porque é categórico: codifique
> (`produto == "cartao"` como 0/1) ou ajuste **um modelo por grupo** (`modelo por grupo: produto`). Sem
> isso, o hazard do cartão e o do consignado saem misturados numa curva média, e a discriminação por
> horizonte cai.

In [ ]:
ui.dd_hz_by.value = "produto"                  # um modelo POR produto
ui.btn_hz.click()
ui.hazard_lt_.summary()

## 6. Aba 4. **Paramétrico**: a forma da maturação e a cauda

A curva empírica só vai até onde a carteira foi observada (idade 35 no painel de referência). Um
contrato de 60 meses precisa de **25 meses de curva que os dados não mostram**. A extensão plana é a
hipótese mínima; a **alternativa defensável** é ajustar uma família ao trecho observado e deixá-la dizer
o que a maturação faz depois.

As famílias, por máxima verossimilhança em tempo discreto ($h_t = 1 - S(t+1)/S(t)$, respeitando censura
e truncagem à esquerda pela própria construção):

| família | forma do hazard | quando faz sentido |
|---|---|---|
| exponencial | constante | referência (sem maturação) |
| **Weibull** | monótono: $k > 1$ cresce, $k < 1$ decresce | maturação de originação; *burn-out* |
| log-normal, log-logística | **corcova** (sobe e depois cai) | crédito ao consumidor com pico de risco no 1º/2º ano |
| Gompertz | exponencial na idade | crescimento acelerado |

**O que olhar.** O **AIC** desempata (`delta_aic` abaixo de 2 = indistinguíveis); o **gráfico contra o
KM** decide. No painel de referência a verdade é Weibull com $k = 1{,}35$: a Weibull deve vencer o
exponencial com folga e recuperar o $k$.

A **emenda**: até a **junção** (última idade cuja base em risco ainda é maior ou igual à base mínima)
manda o KM; dali em diante a família adotada. *Casar o nível* multiplica a cauda pela razão entre a
média empírica e a paramétrica nas últimas idades antes da junção: sem degrau, e o **formato** continua
o da família.

In [ ]:
# = clicar "Ajustar famílias" (todas as cinco), depois "Adotar como curva do estudo" com a emenda
ui.dd_par_by.value = "produto"
ui.sl_par_min_risk.value = 30
ui.sl_par_horizon.value = 60
ui.btn_par.click()

display(ui.param_rank_[["grupo", "posicao", "distribuicao", "parametros", "aic", "delta_aic", "leitura"]])
print("família adotada por padrão (menor AIC agregado):", ui.dd_par_choice.value)

In [ ]:
ui.dd_par_mode.value = "splice"                # empírica + cauda paramétrica
ui.dd_par_choice.value = "weibull"
ui.cb_par_match.value = True
ui.btn_par_adopt.click()

c = ui.model_.curve("cartao")
print(c, "| junção:", c.meta["junction"], "| fator de nível:", round(c.meta["fator_nivel"], 3))
ui.model_.summary()

In [ ]:
# Sem a interface: o mesmo em quatro chamadas
parte = painel_des.by("produto")["cartao"]
rank, modelos = fit_parametric(parte)
j = junction_age(parte, min_at_risk=30)
emendada = splice_curves(kaplan_meier(parte, label="cartao"), modelos["weibull"], junction=j, horizon=60)
print(f"junção = {j} | Weibull k = {modelos['weibull'].params_['shape']:.2f} (verdadeiro {ref.truth['shape']})")
emendada.to_frame().iloc[[11, 23, 35, 47, 59]]

## 7. Aba 5. **Calibração & Ciclo**: nível e ciclo

**Calibração.** O desenho usual: o modelo transversal (scorecard) dá o **nível** da PD de 12 meses por
grupo; a curva de sobrevivência dá o **formato** da maturação. `calibrate_to` desloca o hazard no logit
até a PD acumulada de 12 meses bater com o alvo, preservando a forma. Informe o alvo por grupo (vazio
= não calibra o grupo).

**Ciclo (Vasicek, TTC → PIT).** Convenção de sinal: $z > 0$ ciclo benigno (PD cai), $z < 0$ adverso.
`shift` aplica $\Phi(\Phi^{-1}(PD) - \sqrt{\rho}\,z)$ a cada hazard e é idempotente em $z = 0$;
`conditional` reescala por $1/\sqrt{1-\rho}$ (a lei exata do fator único, a mesma do motor de
capital). A **reversão** dissipa o choque ao longo do horizonte ($z_t = z(1-\text{decay})^t$).

A ordem é sempre **base → calibração → ciclo**, independentemente da ordem dos cliques.

In [ ]:
# = digitar os alvos por grupo e clicar "Calibrar nível"
ui._calib_inputs["cartao"].value = "0.14"      # a PD de 12 meses que saiu do scorecard
ui._calib_inputs["consignado"].value = "0.05"
ui.btn_calib.click()

# = z = -1 (adverso), rho = 0,10, reversão 0,10 e clicar "Condicionar ao ciclo"
ui.fl_z.value = -1.0
ui.fl_rho.value = 0.10
ui.fl_decay.value = 0.10
ui.btn_cycle.click()

print([a["tipo"] for a in ui.model_.adjustments_])     # ['logit_shift', 'vasicek']
pd.DataFrame({
    "base":     {r: c.pd_12m() for r, c in ui.model_base_.curves_.items()},
    "vigente":  {r: c.pd_12m() for r, c in ui.model_.curves_.items()},
}).rename_axis("pd_12m")

> **Sob a 4.966.** Calibrar ao scorecard e depois condicionar ao ciclo é defensável quando o
> scorecard é **TTC** (média de ciclo). Se a PD de 12 meses do scorecard já é **PIT**, condicionar de
> novo conta o ciclo duas vezes. Documente qual é o caso; a validação vai perguntar.

## 8. Aba 6. **Validação**: a curva acerta o nível, ordena e a hipótese sobrevive?

Sempre no **OOT** da partição (a interface avisa quando a validação é *in-sample*). Os contratos são
avaliados a partir da **primeira observação** no painel de validação; censurados antes do horizonte saem
do denominador da discriminação e do decil.

| bloco | o que mede | como ler |
|---|---|---|
| **Calibração por horizonte** | PD acumulada prevista × observada (KM) em 12/24/36, com $z$ de Greenwood | $\lvert z\rvert > 1{,}96$ em todos os horizontes com o mesmo sinal = **nível** errado; sinal trocando = **formato** errado |
| **Discriminação** | AUC/Gini/KS do default em $h$ contra a PD prevista; **C-index** de Harrell | com curva única não há o que ordenar (o ordenamento vem do scorecard); com covariáveis, revise features/baseline |
| **Calibração por decil** | prevista × observada por faixa de PD prevista, IC binomial, **Hosmer-Lemeshow** | faixas fora do IC nas pontas = efeito não linear ou cauda mal emendada |
| **Riscos proporcionais** | o teste da aba Hazard | rejeitar = o efeito muda com a idade: interações |

In [ ]:
# Para a validação fazer sentido, voltamos à curva sem o ciclo (o OOT não foi estressado):
ui.btn_cycle_clear.click()

# = horizontes "12,24,36", horizonte do decil 12 e clicar "Validar"
ui.tx_val_horizons.value = "12,24,36"
ui.sl_val_dec_h.value = 12
ui.btn_val.click()

print("validado em:", ui.validated_on_)
print([(b["bloco"], b["nivel"]) for b in ui.val_blocks_])
ui.backtest_[["grupo", "horizonte", "n_em_risco_h", "pd_prevista", "pd_observada", "ic_inf", "ic_sup", "z", "dentro_do_ic"]]

In [ ]:
display(ui.discrimination_)
print(f"C-index: {ui.c_index_:.3f}")
ui.decile_

### 8.1. A discriminação vem das covariáveis

Uma curva **por grupo** só ordena entre grupos (dois produtos = dois valores de PD). Adote a curva de
**hazard** e a discriminação por horizonte passa a medir o quanto as covariáveis separam o risco.

In [ ]:
ui.btn_hz_adopt.click()          # a curva de hazard (por produto) vira a curva do estudo
ui.btn_val.click()
display(ui.discrimination_[["horizonte", "n", "n_eventos", "auc", "gini", "ks"]])
print(f"C-index: {ui.c_index_:.3f} | riscos proporcionais: "
      f"{[b for b in ui.val_blocks_ if b['bloco'] == 'Riscos proporcionais'][0]['veredito']}")

## 9. Aba 7. **Exportar**: o modelo, a configuração, as tabelas e o MLflow

Três coisas saem daqui:

1. **O modelo em JSON** (`LifetimePD.to_json`): as curvas com a linhagem (método, cauda, calibração,
   ciclo). `LifetimePD.from_json(caminho)` reconstrói o objeto que `ecl_table` consome. No motor de
   hazard, os coeficientes por contrato ficam em `ui.model_.hazard_models_` (persista com `joblib`); o
   JSON leva as curvas de referência.
2. **A configuração em JSON** (`SurvivalConfig`): a curva só é reproduzível se a configuração que a
   gerou viajar junto. É o mesmo objeto que `run_survival_study` consome fora do notebook.
3. **As tabelas em CSV** e o **run no MLflow** (`log_lifetime_pd`: parâmetros, PD 12m/lifetime por
   grupo, curvas, backtest e figuras).

In [ ]:
import tempfile, pathlib
pasta = pathlib.Path(tempfile.mkdtemp())

# = "Salvar modelo"
ui.tx_model_path.value = str(pasta / "pd_lifetime.json")
ui.btn_model_save.click()

# = "Ver JSON da sessão" e "Salvar"
ui.btn_cfg_show.click()
ui.tx_cfg_path.value = str(pasta / "estudo.json")
ui.btn_cfg_save.click()

# = tabela "curvas" e "Salvar CSV"
ui.dd_exp_tabela.value = "curvas"
ui.tx_exp_path.value = str(pasta / "curvas.csv")
ui.btn_exp_salvar.click()

sorted(p.name for p in pasta.iterdir())

In [ ]:
# O modelo volta como um LifetimePD comum e cola na carteira viva
lt = LifetimePD.from_json(str(pasta / "pd_lifetime.json"))
viva = df.groupby("id_contrato").tail(1).copy()
viva["idade"] = (pd.PeriodIndex(viva["dt_ref"], freq="M").asi8
                 - pd.PeriodIndex(viva["safra_origem"], freq="M").asi8)
escorada = lt.apply(viva.head(1000), age_col="idade", term_col="prazo", detail=False)
escorada[["id_contrato", "produto", "idade", "prazo", "pd_12m", "pd_lifetime"]].head()

In [ ]:
# = "Registrar no MLflow" (tracking local em ./mlruns; no Databricks já vem configurado)
ui.tx_mlflow_exp.value = "/tmp/yggdrasil_survival_tutorial"
ui.btn_mlflow.click()
print(ui.out_mlflow_status.value[:200])

## 10. O estudo completo em um clique (e fora do notebook)

O botão **Rodar estudo completo** (aba Painel) monta a `SurvivalConfig` a partir de todos os controles
e encadeia partição → curva → cauda → calibração/ciclo → validação, preenchendo cada aba. Fora da
interface, a mesma configuração roda com `run_survival_study`, o que fecha o ciclo de
reprodutibilidade: a configuração salva na seção 9 reproduz a curva na esteira.

In [ ]:
# = escolher o motor e clicar "Rodar estudo completo"
ui.dd_study_method.value = "km"
ui.btn_run_study.click()
print(ui.out_study_status.value)
ui.study_.summary()

In [ ]:
# Fora da interface: a configuração declarativa
cfg = SurvivalConfig(
    name="pd_lifetime_referencia",
    origin_col="safra_origem", segment_col="produto", term_col="prazo", exposure_col="exposicao",
    by="produto", method="km", horizon=60,
    tail="parametric", distribution="weibull", min_at_risk=30, match_level=True,
    calibrate={"cartao": 0.14, "consignado": 0.05},
    split="origin", split_value=ui.tx_split_date.value,
    backtest_horizons=[12, 24, 36], decile_horizon=12,
)
res = run_survival_study(df, cfg)
print(res.validated_on, "|", [(e, round(s, 2)) for e, s in res.steps])
display(res.summary())

# ...e a config que a interface gerou é o mesmo objeto
cfg_ui = SurvivalConfig.from_json(str(pasta / "estudo.json"))
print(type(cfg_ui).__name__, cfg_ui.method, cfg_ui.tail, cfg_ui.distribution)

## 11. Da curva ao ECL

O `LifetimePD` que sai daqui entra direto em `ecl_table` (tutorial **12**), com o estágio cortando o
horizonte (1 → 12 meses, 2 → lifetime, 3 → ELBE):

```python
from yggdrasil.credit_risk.ecl import ecl_table
res_ecl = ecl_table(carteira, model=ui.model_, lgd="lgd", ead="saldo", stage_col="estagio",
                    elbe="elbe", discount_rate="taxa_efetiva", age_col="idade", term_col="prazo")
```

## Resumo do que a interface entrega

| aba | pergunta | saída |
|---|---|---|
| Painel | o que há na carteira, e onde validar? | tabela de vida, mapa safra × idade, partição DES/OOT |
| Kaplan-Meier | o que a carteira mostrou, com que incerteza, e os segmentos diferem? | curvas com Greenwood, Nelson-Aalen, log-rank |
| Hazard | qual é a curva **de cada contrato**? | regressão em tempo discreto, odds ratio, perfis, riscos proporcionais |
| Paramétrico | qual a forma da maturação, e o que acontece depois da última idade observada? | ranking por AIC, emenda da cauda |
| Calibração & Ciclo | o nível bate com o scorecard, e o ciclo entra como? | `calibrate_to`, `condition` |
| Validação | a curva acerta o nível, ordena, e a hipótese sobrevive? | backtest com $z$, AUC/C-index, decil com HL, PH |
| Exportar | como isso viaja para a esteira e para a auditoria? | JSON do modelo, JSON da config, CSV, MLflow |